In [3]:
import json
import os
import shutil
from collections import Counter
from pathlib import Path

# --- CONFIGURATION ---
# Path to your original 191,961 JSONs
source_annos = '/Users/gdsatishkumar/Downloads/annos'
# Path to your original 191,961 JPGs
source_images = '/Users/gdsatishkumar/Downloads/train/image'

# Destinations for the filtered 144k+ files
output_annos = '/Users/gdsatishkumar/Downloads/annos_updated'
output_images = '/Users/gdsatishkumar/Downloads/images_filtered_all'
# ---------------------

def sync_full_dataset(anno_dir, img_dir, out_anno_dir, out_img_dir):
    os.makedirs(out_anno_dir, exist_ok=True)
    os.makedirs(out_img_dir, exist_ok=True)
    
    path_list = list(Path(anno_dir).glob('*.json'))
    counts = Counter()
    
    # Phase 1: Identify Top 5
    print(f"Phase 1: Analyzing {len(path_list)} files for top categories...")
    for file_path in path_list:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
                for value in data.values():
                    if isinstance(value, dict) and 'category_name' in value:
                        counts[value['category_name']] += 1
        except: continue

    top_5_names = {name for name, count in counts.most_common(5)}
    print(f"Top 5 categories to keep: {', '.join(top_5_names)}")

    # Phase 2: Filter JSONs and Copy Matching Images
    print("Phase 2: Filtering and syncing images (this may take a while)...")
    processed_count = 0
    
    for file_path in path_list:
        try:
            with open(file_path, 'r', encoding='utf-8') as f:
                data = json.load(f)
            
            # Filter the dictionary
            filtered_data = {}
            has_valid_item = False
            
            for key, value in data.items():
                if isinstance(value, dict) and 'category_name' in value:
                    if value['category_name'] in top_5_names:
                        filtered_data[key] = value
                        has_valid_item = True
                else:
                    # Keep metadata (source, pair_id, etc.)
                    filtered_data[key] = value
            
            # Only proceed if the JSON actually contains at least one of the top 5
            if has_valid_item:
                # 1. Save filtered JSON
                out_json_path = os.path.join(out_anno_dir, file_path.name)
                with open(out_json_path, 'w', encoding='utf-8') as f:
                    json.dump(filtered_data, f, indent=None)

                # 2. Find and Copy Image (checking common extensions)
                base_name = file_path.stem
                found_img = False
                for ext in ['.jpg', '.jpeg', '.JPG', '.png']:
                    img_src = os.path.join(img_dir, base_name + ext)
                    if os.path.exists(img_src):
                        shutil.copy2(img_src, os.path.join(out_img_dir, base_name + ext))
                        found_img = True
                        break
                
                if found_img:
                    processed_count += 1
                    if processed_count % 10000 == 0:
                        print(f"Progress: {processed_count} pairs synced...")

        except Exception as e:
            continue

    print(f"Done! Successfully synced {processed_count} JSON/Image pairs to the output folders.")

if __name__ == "__main__":
    sync_full_dataset(source_annos, source_images, output_annos, output_images)

Phase 1: Analyzing 191961 files for top categories...
Top 5 categories to keep: skirt, shorts, short sleeve top, trousers, long sleeve top
Phase 2: Filtering and syncing images (this may take a while)...
Progress: 10000 pairs synced...
Progress: 20000 pairs synced...
Progress: 30000 pairs synced...
Progress: 40000 pairs synced...
Progress: 50000 pairs synced...
Progress: 60000 pairs synced...
Progress: 70000 pairs synced...
Progress: 80000 pairs synced...
Progress: 90000 pairs synced...
Progress: 100000 pairs synced...
Progress: 110000 pairs synced...
Progress: 120000 pairs synced...
Progress: 130000 pairs synced...
Progress: 140000 pairs synced...
Done! Successfully synced 144174 JSON/Image pairs to the output folders.
